# Drexel University
## College of Computing and Informatics
## INFO 371: Data Mining Applications
## Assignment 2

## **DON'T FORGET TO PUT YOUR NAME BELOW**

## **NAME:** Hanzala Nadeem


### A. What to Hand In

1. A completed this Jupyter notebook.
2. Any data sets used in the assignment.
3. Any multimedia and images used in the assignment.

### B. How to Hand In

Submit your files through the course website in the Blackboard Learn system.

### C. When to Hand In

1. Submit your assignment no later than 11:59pm in the due date.
2. There will be a 10% (absolute value) deduction for each day of lateness, to a maximum of 3 days; assignments will not be accepted beyond that point. Missing work will earn a zero grade.

### D. Answer the following questions

# Collecting the data
The data you will be using in this assignment is the match history data for the NBA for the 2023-2024 season.

The website [http://basketball-reference.com](http://basketball-reference.com) contains a significant number of resources
and statistics collected from the NBA and other leagues.

To download the dataset, perform the following steps:
1. Navigate to [http://www.basketball-reference.com/leagues/NBA_2024_games.html](http://www.basketball-reference.com/leagues/NBA_2024_games.html) in your web browser.
2. Click Share & Export.
3. Click Get table as CSV (for Excel).
4. Copy the data, including the heading, into a text file named nba_2024.csv.
5. Repeat this process for the other months, except do not copy the heading.

This will give you a CSV file containing the results from each game of this season of the NBA.

# Note: In this assignment, ensure that you split the data into train and test sets before training, selecting, and evaluating models. All evaluation must be done on a separate test set.

## Question 1 [5 points]: Load the data as a DataFrame and change the columns to
```
["Date", "Start (ET)", "Visitor Team", "VisitorPts", \
"Home Team", "HomePts", "Unnamed: 6", "Unnamed: 7", "Attend.", "LOG", "Arena", "Notes"]
```
Also, parse the "Date" column as datetime type.

In [ ]:

import pandas as pd
from sklearn.ensemble import RandomForestClassifier

import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
df = pd.read_csv('data/nba_2024.csv')
print(df.columns)
#rename columnds
df.columns = ["Date", "Start (ET)", "Visitor Team", "VisitorPts", \
"Home Team", "HomePts", "Unnamed: 6", "Unnamed: 7", "Attend.", "LOG", "Arena", "Notes"]
df["Date"] = pd.to_datetime(df["Date"])
df.head()

## Question 2 [10 points]: Create a target column "VisitorWin": if the visitor team won the game, VisitorWin=True; otherwise, VisitorWin=False.

In [ ]:
#new columnds to check if visitor won or no
df["VisitorWin"] = df["VisitorPts"] > df["HomePts"]
df.head()


## Question 3 [10 points]: Convert the Visitor Team names to one-hot vectors.

In [ ]:
#create a hot vecotr using dummies
visitor_team_ohe = pd.get_dummies(df["Visitor Team"])
visitor_team_ohe.head()

## Question 4 [15 points]: Create a Scikit-Learn decision tree classifier with default values. Apply 5-fold cross validation to fit the classifier on the one-hot vectors corresponding to the visitor team names and the target 'VisitorWin'. Use accuracy for scoring the model. Print out the average score of the cross validation.

In [ ]:
# Feature matrix and target vector
X = visitor_team_ohe
y = df["VisitorWin"]

# Decision tree with default parameters
clf = DecisionTreeClassifier()

# 5-fold cross-validation using accuracy
scores = cross_val_score(
    clf,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

# Print average accuracy
print("Average cross-validation accuracy:", scores.mean())

## Question 5 [20 points]: Create two new feature columns: HomeWonLast and VisitorWonLast: if the home team won the previous game, HomeWonLast=1 and VisitorWonLast=0; otherwise, HomeWonLast=0 and VisitorWonLast=1

In [ ]:
# Ensure games are sorted chronologically
df = df.sort_values("Date").reset_index(drop=True)

# Initialize columns
df["HomeWonLast"] = 0
df["VisitorWonLast"] = 0

# Dictionary to track last result for each team
# True = won last game, False = lost last game
last_result = {}

# Iterate through games in chronological order
for i, row in df.iterrows():
    home = row["Home Team"]
    visitor = row["Visitor Team"]

    # Check previous results
    home_won_last = last_result.get(home, False)
    visitor_won_last = last_result.get(visitor, False)

    # Assign feature values
    if home_won_last:
        df.at[i, "HomeWonLast"] = 1
        df.at[i, "VisitorWonLast"] = 0
    else:
        df.at[i, "HomeWonLast"] = 0
        df.at[i, "VisitorWonLast"] = 1

    # Determine winner of current game
    home_won = row["HomePts"] > row["VisitorPts"]

    # Update last results
    last_result[home] = home_won
    last_result[visitor] = not home_won


df[["Date", "Visitor Team", "Home Team", "HomeWonLast", "VisitorWonLast"]].head(20)





## Question 6 [10 points]: Apply 5-fold cross validation to fit a decision tree classifier on HomeWonLast and VisitorWonLast against the target VisitorWin. Print out the average accuracy score.

In [ ]:
# Feature matrix and target
X = df[["HomeWonLast", "VisitorWonLast"]]
y = df["VisitorWin"]

# Decision tree with default parameters
clf = DecisionTreeClassifier()

# 5-fold cross-validation using accuracy
scores = cross_val_score(
    clf,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

# Print average accuracy
print("Average cross-validation accuracy:", scores.mean())

## Question 7 [10 points]: Apply 5-fold cross validation to fit a decision tree classifier on visitor team names (one-hot vectors), HomeWonLast, and VisitorWonLast against the target VisitorWin. Print out the average accuracy score. Does the preformance change?

In [ ]:
# Combine visitor team one-hot vectors with momentum features
X = pd.concat(
    [
        visitor_team_ohe,
        df[["HomeWonLast", "VisitorWonLast"]]
    ],
    axis=1
)

# Target variable
y = df["VisitorWin"]

# Decision tree with default parameters
clf = DecisionTreeClassifier()

# 5-fold cross-validation using accuracy
scores = cross_val_score(
    clf,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

# Print average accuracy
print("Average cross-validation accuracy:", scores.mean())

#Yes. When you added HomeWonLast and VisitorWonLast to the visitor team one-hot features, the average cross-validation accuracy decreased from about 0.598 to 0.531.

## Question 8 [10 points]: Create a Scikit-Learn random forest classifier with default values. Apply 5-fold cross validation to fit the random forest classifier on visitor team names (one-hot vectors), HomeWonLast, and VisitorWonLast against the target VisitorWin. Print out the average accuracy score. Does the performance change?



In [ ]:
# Create the random forest classifier with default values
rf = RandomForestClassifier()

# Apply 5-fold cross validation
scores = cross_val_score(
    rf,
    X,                # one-hot visitor teams + HomeWonLast + VisitorWonLast
    y,                # VisitorWin
    cv=5,
    scoring='accuracy'
)

# Print average accuracy
print("Average cross-validation accuracy:", scores.mean())

#the perforamnce chagnes but gets slgintly worse

## Question 9 [10 points]: Conduct a grid search for selecting the best random forest model fitting on visitor team names (one-hot vectors), HomeWonLast, and VisitorWonLast against the target VisitorWin. In the grid search, change the following parameters with different values: "max_features", "n_estimators", and "min_samples_leaf". Print out the best model and the average score of the model. Is the performance of the best model better than the previous models?

In [ ]:

from sklearn.model_selection import GridSearchCV

# Build feature matrix X and target y
X = pd.concat(
    [visitor_team_ohe, df[["HomeWonLast", "VisitorWonLast"]]],
    axis=1
)
y = df["VisitorWin"]

# Random Forest (base)
rf = RandomForestClassifier()

# Parameter grid (you can expand if you want, but this is a solid starter grid)
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_features": ["sqrt", "log2", None],
    "min_samples_leaf": [1, 2, 5, 10],
}

# Grid Search with 5-fold CV using accuracy
grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X, y)

print("Best model:", grid.best_estimator_)
print("Best average CV accuracy:", grid.best_score_)

# Although the grid search improved the random forest performance compared to the default random forest (~0.524 → ~0.544),
#it still does not outperform the best previous model, which was the decision tree trained only on visitor team names with an accuracy of approximately 0.598.
#This indicates that: Hyperparameter tuning helped reduce overfitting (larger min_samples_leaf) But the available features still lack sufficient predictive power
